In [1]:
import argparse
import yaml
import os
import logging
import shutil
import numpy as np
from PIL import Image 
# logger = logging.getLogger()
handler = logging.StreamHandler()
handler.setLevel(logging.INFO)
# logger.addHandler(handler)

import torch
import torch.optim as optim
import torchvision.transforms as transforms
from diffusers import DDIMScheduler, DDIMInverseScheduler
from datasets import load_dataset
from diffusers.utils.torch_utils import randn_tensor

from main.wmpatch import GTWatermark, GTWatermarkMulti
from main.utils import *
from loss.loss import LossProvider
from loss.pytorch_ssim import ssim

## Necessary Setup for All Sections

In [2]:
logging.info(f'===== Load Config =====')
device = torch.device('cuda')
with open('./example/config/config.yaml', 'r') as file:
    cfgs = yaml.safe_load(file)
logging.info(cfgs)

INFO:root:===== Load Config =====
INFO:root:{'method': 'ZoDiac', 'save_img': './example/output/', 'model_id': 'stabilityai/stable-diffusion-xl-base-1.0', 'cache_dir': '/project/pi_huiguan_umass_edu/lijun/lijun_data/diffusion/model/', 'gen_seed': 0, 'empty_prompt': True, 'w_type': 'single', 'w_channel': 3, 'w_radius': 10, 'w_seed': 10, 'start_latents': 'init_w', 'iters': 100, 'save_iters': [100], 'loss_weights': [10.0, 0.1, 1.0, 0.0], 'ssim_threshold': 0.92, 'detect_threshold': 0.9}


In [3]:
if 'xl' in cfgs['model_id']:
    from main.wmdiffusionxl import WatermarkStableDiffusionPipeline
else:
    from main.wmdiffusion import WatermarkStableDiffusionPipeline

In [5]:
logging.info(f'===== Init Pipeline =====')
if cfgs['w_type'] == 'single':
    wm_pipe = GTWatermark(device, w_channel=cfgs['w_channel'], w_radius=cfgs['w_radius'], generator=torch.Generator(device).manual_seed(cfgs['w_seed']))
elif cfgs['w_type'] == 'multi':
    wm_pipe = GTWatermarkMulti(device, w_settings=cfgs['w_settings'], generator=torch.Generator(device).manual_seed(cfgs['w_seed']))

scheduler = DDIMScheduler.from_pretrained(cfgs['model_id'], subfolder="scheduler", cache_dir=cfgs['cache_dir'])
pipe = WatermarkStableDiffusionPipeline.from_pretrained(cfgs['model_id'], scheduler=scheduler, cache_dir=cfgs['cache_dir'])
pipe.set_progress_bar_config(disable=True)

INFO:root:===== Init Pipeline =====
DEBUG:urllib3.connectionpool:Resetting dropped connection: huggingface.co
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /stabilityai/stable-diffusion-xl-base-1.0/resolve/main/scheduler/scheduler_config.json HTTP/1.1" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "GET /api/models/stabilityai/stable-diffusion-xl-base-1.0 HTTP/1.1" 200 6336
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /stabilityai/stable-diffusion-xl-base-1.0/resolve/main/model_index.json HTTP/1.1" 200 0


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [6]:
pipe = pipe.to('cuda')

In [7]:
imagename = 'pepper.tiff'
gt_img_tensor = get_img_tensor(f'./example/input/{imagename}', device)
wm_path = cfgs['save_img']

DEBUG:PIL.Image:Importing BlpImagePlugin
DEBUG:PIL.Image:Importing BmpImagePlugin
DEBUG:PIL.Image:Importing BufrStubImagePlugin
DEBUG:PIL.Image:Importing CurImagePlugin
DEBUG:PIL.Image:Importing DcxImagePlugin
DEBUG:PIL.Image:Importing DdsImagePlugin
DEBUG:PIL.Image:Importing EpsImagePlugin
DEBUG:PIL.Image:Importing FitsImagePlugin
DEBUG:PIL.Image:Importing FliImagePlugin
DEBUG:PIL.Image:Importing FpxImagePlugin
DEBUG:PIL.Image:Image: failed to import FpxImagePlugin: No module named 'olefile'
DEBUG:PIL.Image:Importing FtexImagePlugin
DEBUG:PIL.Image:Importing GbrImagePlugin
DEBUG:PIL.Image:Importing GifImagePlugin
DEBUG:PIL.Image:Importing GribStubImagePlugin
DEBUG:PIL.Image:Importing Hdf5StubImagePlugin
DEBUG:PIL.Image:Importing IcnsImagePlugin
DEBUG:PIL.Image:Importing IcoImagePlugin
DEBUG:PIL.Image:Importing ImImagePlugin
DEBUG:PIL.Image:Importing ImtImagePlugin
DEBUG:PIL.Image:Importing IptcImagePlugin
DEBUG:PIL.Image:Importing JpegImagePlugin
DEBUG:PIL.Image:Importing Jpeg2KImageP

In [8]:
1 - watermark_prob_new(gt_img_tensor, pipe, wm_pipe)

The config attributes {'sample_max_value': 1.0, 'skip_prk_steps': True} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.


  0%|          | 0/50 [00:00<?, ?it/s]

/work/pi_huiguan_umass_edu/lijunzhang/.conda/envs/diffusion/lib/python3.10/site-packages/torch/utils/checkpoint.py:31: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn("None of the inputs have requires_grad=True. Gradients will be None")


0.0957347753207558

## Image Watermarking

In [95]:
# Step 1: Get init noise
init_latents_approx = get_init_latent(gt_img_tensor, pipe, scheduler)
pipe.set_progress_bar_config(disable=True)

The config attributes {'interpolation_type': 'linear', 'skip_prk_steps': True, 'use_karras_sigmas': False} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.


In [96]:
# Step 2: prepare training
init_latents = init_latents_approx.detach().clone()
init_latents.requires_grad = True
# optimizer = optim.Adam([init_latents], lr=0.01)
# lr_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30,80], gamma=0.3)
optimizer = optim.AdamW([init_latents], lr=0.01, weight_decay=0)
lr_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30,80,150], gamma=0.3) 

totalLoss = LossProvider(cfgs['loss_weights'], device)
loss_lst = [] 

In [98]:
# Step 3: train the init latents
for i in range(cfgs['iters']):
    logging.info(f'iter {i}:')
    init_latents_wm = wm_pipe.inject_watermark(init_latents)
    if cfgs['empty_prompt']:
        pred_img_tensor = pipe('', guidance_scale=1.0, num_inference_steps=10, output_type='tensor', use_trainable_latents=True, init_latents=init_latents_wm).images
    else:
        pred_img_tensor = pipe(prompt, num_inference_steps=10, output_type='tensor', use_trainable_latents=True, init_latents=init_latents_wm).images
    loss = totalLoss(pred_img_tensor, gt_img_tensor, init_latents_wm, wm_pipe)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    lr_scheduler.step()

    loss_lst.append(loss.item())
    # save watermarked image
    if (i+1) in cfgs['save_iters']:
        path = os.path.join(wm_path, f"{imagename.split('.')[0]}_{i+1}.png")
        save_img(path, pred_img_tensor, pipe)
torch.cuda.empty_cache()

INFO:root:iter 0:
INFO:root:Watermark 0.0000, Image 0.4336, Perp 1.1333, SSIM 0.5630 Total Loss 2.1299
INFO:root:iter 1:
INFO:root:Watermark 0.0000, Image 0.4223, Perp 1.1082, SSIM 0.5556 Total Loss 2.0861
INFO:root:iter 2:
INFO:root:Watermark 0.0000, Image 0.4098, Perp 1.0935, SSIM 0.5498 Total Loss 2.0531
INFO:root:iter 3:
INFO:root:Watermark 0.0000, Image 0.3943, Perp 1.0831, SSIM 0.5456 Total Loss 2.0229
INFO:root:iter 4:
INFO:root:Watermark 0.0000, Image 0.3863, Perp 1.0585, SSIM 0.5359 Total Loss 1.9806
INFO:root:iter 5:
INFO:root:Watermark 0.0000, Image 0.3841, Perp 1.0447, SSIM 0.5361 Total Loss 1.9649
INFO:root:iter 6:
INFO:root:Watermark 0.0000, Image 0.3862, Perp 1.0407, SSIM 0.5383 Total Loss 1.9653
INFO:root:iter 7:
INFO:root:Watermark 0.0000, Image 0.4046, Perp 1.0653, SSIM 0.5413 Total Loss 2.0112
INFO:root:iter 8:
INFO:root:Watermark 0.0000, Image 0.4119, Perp 1.0552, SSIM 0.5419 Total Loss 2.0090
INFO:root:iter 9:
INFO:root:Watermark 0.0000, Image 0.4054, Perp 1.0462, 

KeyboardInterrupt: 

## Postprocessing with Adaptive Enhancement

In [99]:
# hyperparameter
ssim_threshold = cfgs['ssim_threshold']

In [100]:
wm_img_path = os.path.join(wm_path, f"{imagename.split('.')[0]}_{cfgs['save_iters'][-1]}.png")
wm_img_tensor = get_img_tensor(wm_img_path, device)
ssim_value = ssim(wm_img_tensor, gt_img_tensor).item()
logging.info(f'Original SSIM {ssim_value}')

DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 65536
INFO:root:Original SSIM 0.5720533132553101


In [101]:
def binary_search_theta(threshold, lower=0., upper=1., precision=1e-6, max_iter=1000):
    for i in range(max_iter):
        mid_theta = (lower + upper) / 2
        img_tensor = (gt_img_tensor-wm_img_tensor)*mid_theta+wm_img_tensor
        ssim_value = ssim(img_tensor, gt_img_tensor).item()

        if ssim_value <= threshold:
            lower = mid_theta
        else:
            upper = mid_theta
        if upper - lower < precision:
            break
    return lower

optimal_theta = binary_search_theta(ssim_threshold, precision=0.01)
logging.info(f'Optimal Theta {optimal_theta}')

img_tensor = (gt_img_tensor-wm_img_tensor)*optimal_theta+wm_img_tensor

ssim_value = ssim(img_tensor, gt_img_tensor).item()
psnr_value = compute_psnr(img_tensor, gt_img_tensor)

det_prob = 1 - watermark_prob_new(img_tensor, pipe, wm_pipe)

path = os.path.join(wm_path, f"{os.path.basename(wm_img_path).split('.')[0]}_SSIM{ssim_threshold}.png")
save_img(path, img_tensor, pipe)
logging.info(f'SSIM {ssim_value}, PSNR, {psnr_value}, Detect Prob: {det_prob} after postprocessing')

INFO:root:Optimal Theta 0.6328125
The config attributes {'interpolation_type': 'linear', 'skip_prk_steps': True, 'use_karras_sigmas': False} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.
INFO:root:SSIM 0.9176234006881714, PSNR, 24.744992417495904, Detect Prob: 0.999257030476602 after postprocessing


## Attack Watermarked Image with Individual Attacks

In [37]:
from main.wmattacker import *
from main.attdiffusion import ReSDPipeline

logging.info(f'===== Init Attackers =====')
att_pipe = ReSDPipeline.from_pretrained("stabilityai/stable-diffusion-2-1", cache_dir=cfgs['cache_dir'], torch_dtype=torch.float16, revision="fp16")
att_pipe.set_progress_bar_config(disable=True)
att_pipe.to(device)

attackers = {
    'diff_attacker_60': DiffWMAttacker(att_pipe, batch_size=5, noise_step=60, captions={}),
    'cheng2020-anchor_3': VAEWMAttacker('cheng2020-anchor', quality=3, metric='mse', device=device),
    'bmshj2018-factorized_3': VAEWMAttacker('bmshj2018-factorized', quality=3, metric='mse', device=device),
    'jpeg_attacker_50': JPEGAttacker(quality=50),
    'rotate_90': RotateAttacker(degree=90),
    'brightness_0.5': BrightnessAttacker(brightness=0.5),
    'contrast_0.5': ContrastAttacker(contrast=0.5),
    'Gaussian_noise': GaussianNoiseAttacker(std=0.05),
    'Gaussian_blur': GaussianBlurAttacker(kernel_size=5, sigma=1),
    'bm3d': BM3DAttacker(),
}

INFO:root:===== Init Attackers =====
DEBUG:urllib3.connectionpool:Resetting dropped connection: huggingface.co
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "GET /api/models/stabilityai/stable-diffusion-2-1/revision/fp16 HTTP/1.1" 200 4528
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /stabilityai/stable-diffusion-2-1/resolve/fp16/model_index.json HTTP/1.1" 200 0
text_encoder/model.safetensors not found


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Diffuse attack initialized with noise step 60 and use prompt 0


In [102]:
logging.info(f'===== Start Attacking... =====')

post_img = os.path.join(wm_path, f"{imagename.split('.')[0]}_{cfgs['save_iters'][-1]}_SSIM{ssim_threshold}.png")
for attacker_name, attacker in attackers.items():
    print(f'Attacking with {attacker_name}')
    os.makedirs(os.path.join(wm_path, attacker_name), exist_ok=True)
    att_img_path = os.path.join(wm_path, attacker_name, os.path.basename(post_img))
    attackers[attacker_name].attack([post_img], [att_img_path])

INFO:root:===== Start Attacking... =====


Attacking with diff_attacker_60


0it [00:00, ?it/s]

DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 65536


Attacking with cheng2020-anchor_3


...

Attacking with Gaussian_blur


0it [00:00, ?it/s]

Attacking with bm3d


0it [00:00, ?it/s]

DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 65536


## Attack Watermarked Image with Combined Attacks

In [ ]:
from main.wmattacker import *
from main.attdiffusion import ReSDPipeline

case_list = ['w/ rot', 'w/o rot']

logging.info(f'===== Init Attackers =====')
att_pipe = ReSDPipeline.from_pretrained("stabilityai/stable-diffusion-2-1", torch_dtype=torch.float16, revision="fp16")
att_pipe.set_progress_bar_config(disable=True)
att_pipe.to(device)

In [ ]:
post_img = os.path.join(wm_path, f"{imagename.split('.')[0]}_{cfgs['save_iters'][-1]}_SSIM{ssim_threshold}.png")

for case in case_list:
    print(f'Case: {case}')
    if case == 'w/ rot':
        attackers = {
        'diff_attacker_60': DiffWMAttacker(att_pipe, batch_size=5, noise_step=60, captions={}),
        'cheng2020-anchor_3': VAEWMAttacker('cheng2020-anchor', quality=3, metric='mse', device=device),
        'bmshj2018-factorized_3': VAEWMAttacker('bmshj2018-factorized', quality=3, metric='mse', device=device),
        'jpeg_attacker_50': JPEGAttacker(quality=50),
        'rotate_90': RotateAttacker(degree=90),
        'brightness_0.5': BrightnessAttacker(brightness=0.5),
        'contrast_0.5': ContrastAttacker(contrast=0.5),
        'Gaussian_noise': GaussianNoiseAttacker(std=0.05),
        'Gaussian_blur': GaussianBlurAttacker(kernel_size=5, sigma=1),
        'bm3d': BM3DAttacker(),
        }
        multi_name = 'all'
    elif case == 'w/o rot':
        attackers = {
        'diff_attacker_60': DiffWMAttacker(att_pipe, batch_size=5, noise_step=60, captions={}),
        'cheng2020-anchor_3': VAEWMAttacker('cheng2020-anchor', quality=3, metric='mse', device=device),
        'bmshj2018-factorized_3': VAEWMAttacker('bmshj2018-factorized', quality=3, metric='mse', device=device),
        'jpeg_attacker_50': JPEGAttacker(quality=50),
        'brightness_0.5': BrightnessAttacker(brightness=0.5),
        'contrast_0.5': ContrastAttacker(contrast=0.5),
        'Gaussian_noise': GaussianNoiseAttacker(std=0.05),
        'Gaussian_blur': GaussianBlurAttacker(kernel_size=5, sigma=1),
        'bm3d': BM3DAttacker(),
        }
        multi_name = 'all_norot'
        
    
    os.makedirs(os.path.join(wm_path, multi_name), exist_ok=True)
    att_img_path = os.path.join(wm_path, multi_name, os.path.basename(post_img))
    for i, (attacker_name, attacker) in enumerate(attackers.items()):
        print(f'Attacking with {attacker_name}')
        if i == 0:
            attackers[attacker_name].attack([post_img], [att_img_path], multi=True)
        else:
            attackers[attacker_name].attack([att_img_path], [att_img_path], multi=True)

## Detect Watermark

In [39]:
post_img = os.path.join(wm_path, f"{imagename.split('.')[0]}_{cfgs['save_iters'][-1]}_SSIM{ssim_threshold}.png")

attacker_list = ['diff_attacker_60', 'cheng2020-anchor_3', 'bmshj2018-factorized_3', 'jpeg_attacker_50', 
             'brightness_0.5', 'contrast_0.5', 'Gaussian_noise', 'Gaussian_blur', 'rotate_90', 'bm3d', 
             'all', 'all_norot']

In [93]:
logging.info(f'===== Testing the Watermarked Images {post_img} =====')
det_prob = 1 - watermark_prob_new(post_img, pipe, wm_pipe)
logging.info(f'Watermark Presence Prob.: {det_prob}')

INFO:root:===== Testing the Watermarked Images ./example/output/pepper_100_SSIM0.92.png =====
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 65536
The config attributes {'interpolation_type': 'linear', 'skip_prk_steps': True, 'use_karras_sigmas': False} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.
INFO:root:Watermark Presence Prob.: 0.9946416101152896


In [103]:
logging.info(f'===== Testing the Attacked Watermarked Images =====')
for attacker_name in attacker_list:
    if not os.path.exists(os.path.join(wm_path, attacker_name)):
        logging.info(f'Attacked images under {attacker_name} not exist.')
        continue
        
    logging.info(f'=== Attacker Name: {attacker_name} ===')
    det_prob = 1 - watermark_prob_new(os.path.join(wm_path, attacker_name, os.path.basename(post_img)), pipe, wm_pipe)
    logging.info(f'Watermark Presence Prob.: {det_prob}')

INFO:root:===== Testing the Attacked Watermarked Images =====
INFO:root:=== Attacker Name: diff_attacker_60 ===
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 65536
The config attributes {'interpolation_type': 'linear', 'skip_prk_steps': True, 'use_karras_sigmas': False} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.
INFO:root:Watermark Presence Prob.: 0.9994622848189303
INFO:root:=== Attacker Name: cheng2020-anchor_3 ===
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 41 65536
The config attributes {'interpolation_type': 'linear', 'skip_prk_steps': True, 'use_karras_sigmas': False} were passed to DDIMInverseScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.
INFO:root:Watermark Presence Prob.: 0.9987181518474056
INFO:root:=== Attacker Name: bmshj2018-fac